In [6]:
import re

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [18]:
def snake_case(name: str) -> str:
    """Convert column names to snake_case."""
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    return name.lower().strip("_")


def process_campaign_file(
    input_path: str,
    output_path: str,
    bleeder_threshold: float = 42.0,
    good_acos_threshold: float = 20.0,
) -> pd.DataFrame:
    # Load CSV
    df = pd.read_csv(input_path)

    # Error handling
    if df.empty:
        raise ValueError("Input CSV is empty.")

    # Convert column names to snake_case
    df.columns = [snake_case(col) for col in df.columns]

    # Required columns check
    required_cols = [
        "campaigns",
        "start_date",
        "impressions",
        "clicks",
        "spend_usd",
        "orders",
        "sales_usd",
        "acos",
    ]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # --- Add / override columns ---
    df["portfolio"] = "Name Change Kit"
    df["budget_impressions"] = ""  # following instructions
    df["budget"] = ""  # user requested empty string
    df["lower_bids_for_bleeders_w_no_sales"] = ""
    df["lower_bids_for_targets_w_acos_higher_than"] = f"{bleeder_threshold}%"
    df["increase_bids_to_targets_w_low_clicks"] = ""
    df["increase_bids_for_targets_with_good_acos"] = ""
    df["change_campaign_budget"] = ""
    df["pause_campaign"] = ""
    df["category"] = ""
    df["action"] = ""

    # Convert ACOS decimal → percentage float (0.42 → 42.0)
    df["acos"] = df["acos"].astype(float) * 100

    # Ensure required numeric columns are numeric
    numeric_cols = ["clicks", "orders", "acos"]
    for col in numeric_cols:
        if not np.issubdtype(df[col].dtype, np.number):
            raise TypeError(f"Column {col} must be numeric.")

    # ---------------------------
    # CATEGORY + ACTION LOGIC
    # ---------------------------

    # Priority 1: Bleeder – no orders
    mask_bleeder_no_orders = (df["orders"] == 0) & (df["clicks"] >= 16)

    df.loc[mask_bleeder_no_orders, "category"] = "bleeder - no orders"

    # 16–18 clicks -> reduce 5%
    df.loc[mask_bleeder_no_orders & df["clicks"].between(16, 18), "action"] = (
        "reduce bid by 5%"
    )

    # 19–25 clicks -> reduce 10%
    df.loc[mask_bleeder_no_orders & df["clicks"].between(19, 25), "action"] = (
        "reduce bid by 10%"
    )

    # >25 clicks -> pause
    df.loc[mask_bleeder_no_orders & (df["clicks"] > 25), "action"] = "pause"

    # Priority 2: Bleeder ≥ 1 order AND ACOS > threshold
    mask_bleeder_ge1 = (df["orders"] >= 1) & (df["acos"] > bleeder_threshold)

    df.loc[mask_bleeder_ge1, "category"] = "bleeder >= 1 order"

    # Reduce bid by the difference
    df.loc[mask_bleeder_ge1, "action"] = (
        "reduce bid by " + (df["acos"] - bleeder_threshold).round(2).astype(str) + "%"
    )

    # Priority 3: Low clicks (<=1)
    mask_low_click = df["clicks"] <= 1

    df.loc[mask_low_click, "category"] = "low click"
    df.loc[mask_low_click, "action"] = "increase bid by 10%"

    # Priority 4: Good ACOS (>=1 order AND ACOS < threshold)
    mask_good_acos = (df["orders"] >= 1) & (df["acos"] < good_acos_threshold)

    df.loc[mask_good_acos, "category"] = "good acos"
    df.loc[mask_good_acos, "action"] = "increase bid by x%"

    # select and order final columns
    key_cols = [
        "campaigns",
        "start_date",
        "portfolio",
        "budget_impressions",
        "clicks",
        "spend_usd",
        "orders",
        "sales_usd",
        "acos",
        "category",
        "action",
    ]
    df = df[key_cols].sort_values(by="spend_usd", ascending=False)
    df["cost_per_order"] = (df["spend_usd"] / df["orders"]).round(2)

    # Save output
    df.to_csv(output_path, index=False)

    return df

In [19]:
file = "december_3_2025"
processed_df = process_campaign_file(
    input_path=f"../inputs/{file}.csv",
    output_path=f"../outputs/{file}.csv",
    bleeder_threshold=42.0,
    good_acos_threshold=20.0,
)
processed_df

,campaigns,start_date,portfolio,budget_impressions,clicks,spend_usd,orders,sales_usd,acos,category,action,cost_per_order
3,SP - Manual - ASIN - Direct Competitors,9/5/25,Name Change Kit,,439,885.11,26,2314.0,38.25,,,34.04
4,SP - Manual - Exact - Main KWs,9/5/25,Name Change Kit,,198,423.37,11,979.0,43.25,bleeder >= 1 order,reduce bid by 1.25%,38.49
5,SP - Auto - Catch All - All Products,8/28/25,Name Change Kit,,320,182.49,2,178.0,102.52,bleeder >= 1 order,reduce bid by 60.52%,91.24
0,SP - Auto - TOS - All Products,9/5/25,Name Change Kit,,68,58.22,3,267.0,21.81,,,19.41
2,SP - Manual - Broad - Main KWs,9/5/25,Name Change Kit,,21,39.16,1,89.0,44.00,bleeder >= 1 order,reduce bid by 2.0%,39.16
1,SP - Auto - Low Bid - All Products,9/5/25,Name Change Kit,,4,1.45,0,0.0,0.00,,,inf


In [ ]:
# sell for 89
# Cost is $45 per unit including amazon shipping but excluding marketing cost
# So, the breakeven rate for markeitng cost is 89 - 45 = 44
print(885.11 / 26)
print(423 / 11)
print(182.49 / 2)
print(58.22 / 3)

34.042692307692306
38.45454545454545
91.245
19.406666666666666


In [ ]:
25